In [1]:
!pip install numpy
!pip install torch
!pip install -q tqdm

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch_xla
from tqdm import tqdm

In [3]:
class PINN(nn.Module):
    def __init__(self, layers=None):
        super().__init__()
        if layers is None:
            layers = [2, 50, 50, 50, 50, 1]
        self.activation = nn.Tanh()
        self.layers = nn.ModuleList()

        for i in range(len(layers)-1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        for layer in self.layers[:-1]:
            inputs = self.activation(layer(inputs))

        return self.layers[-1](inputs)

In [4]:
def compute_wave_equation_residue(model, x, t, c):
    psi = model(x, t)
    psi_t = torch.autograd.grad(psi, t, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psi_t_t = torch.autograd.grad(psi_t, t, torch.ones_like(psi_t), create_graph=True, retain_graph=True)[0]
    psi_x = torch.autograd.grad(psi, x, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psi_x_x = torch.autograd.grad(psi_x, x, torch.ones_like(psi_x), create_graph=True, retain_graph=True)[0]

    residue = psi_x_x - 1 / (c*c) * psi_t_t
    return (residue ** 2).mean()

In [5]:
def analytical_solution(x, t, c, A=1., L=1.):
    return A * np.sin(np.pi*x/L) * np.cos(np.pi*x*t/L)

In [6]:
device = torch_xla.device()

In [7]:
n_colloc, n_bc = 5000, 200
epochs = 5000
A = 1.
L = 1.

In [8]:
x_colloc = torch.rand(n_colloc, 1, requires_grad=True).to(device)

In [9]:
t_colloc = torch.rand(n_colloc, 1, requires_grad=True).to(device)

In [10]:
# initial condition
x_ic = torch.rand(n_bc, 1).to(device)
t_ic = torch.zeros(n_bc, 1).to(device)
psi_ic = A * torch.sin(np.pi * x_ic / L)

In [11]:
# boundary conditions
x_bc = torch.cat([torch.zeros(n_bc // 2, 1), torch.ones(n_bc // 2, 1) * L]).to(device)
t_bc = torch.rand(n_bc, 1).to(device)
psi_bc = torch.zeros(n_bc, 1).to(device)

In [12]:
pinn = PINN().to(device)
optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)

In [13]:
losses_pde = []
losses_ic = []
losses_bc = []

In [ ]:
for epoch in tqdm(range(epochs)):
    optimizer.zero_grad()

    loss_pde = compute_wave_equation_residue(pinn, x_colloc, t_colloc, c=1.)
    loss_ic = nn.functional.mse_loss(pinn(x_ic, t_ic), psi_ic)
    loss_bc = nn.functional.mse_loss(pinn(x_bc, t_bc), psi_bc)

    loss = loss_pde + 10 * loss_ic + 10 * loss_bc

    loss.backward()
    optimizer.step()

    losses_pde.append(loss_pde.item())
    losses_ic.append(loss_ic.item())
    losses_bc.append(loss_bc.item())

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch}: {loss.item():.2f}")

  1%|          | 32/5000 [30:40<155:40:10, 112.80s/it]